In [ ]:
import torch
import numpy as np
from sklearn.metrics import accuracy_score

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iris Classification with Multiple Supervision Regimes

This notebook demonstrates how to express a multi-class classification task in `sltn` using first-order formulas and differentiable fuzzy semantics.

We consider the Iris dataset, where each example is represented by four real-valued features and belongs to one of three classes. The three classes are represented by unary predicates:

- $A(x)$: example $x$ belongs to class A;
- $B(x)$: example $x$ belongs to class B;
- $C(x)$: example $x$ belongs to class C.

The predicates are grounded by a shared neural model. Logical formulas specify the supervision signal, and training maximizes the satisfaction of the selected knowledge base.

The notebook illustrates three types of supervision:

1. full supervision, where each class is supervised directly;
2. supervision for a subset of classes, together with a quantitative estimate for another class;
3. partial supervision with ambiguous labels, expressed through disjunctions.

## Dataset: Iris

The Iris dataset is loaded from local CSV files. Each row contains four numerical features and a class label.

The feature matrix is converted to a tensor, while labels are converted to integer class indices:

- label `0` corresponds to class A;
- label `1` corresponds to class B;
- label `2` corresponds to class C.

In [ ]:
import pandas as pd
from pathlib import Path

train_data = pd.read_csv(Path("../datasets/iris_training.csv"))
test_data  = pd.read_csv(Path("../datasets/iris_test.csv"))

train_labels = train_data.pop("species")
test_labels  = test_data.pop("species")

train_data   = torch.tensor(train_data.to_numpy()).float()
test_data    = torch.tensor(test_data.to_numpy()).float()
train_labels = torch.tensor(train_labels.to_numpy()).long()
test_labels  = torch.tensor(test_labels.to_numpy()).long()

## 1. Signature

A `Signature` declares the symbolic vocabulary used to write formulas.

In this notebook, the signature contains:

- the sort `Item`, representing an Iris sample with four real-valued features;
- three unary predicates `A`, `B`, and `C`, representing the three classes;
- a nullary predicate `T` (i.e. a proposition), used as a quantitative truth-value prior in one supervision regime;
- variables for fully supervised class-specific examples;
- variables for examples with ambiguous labels, such as examples known to belong to either class A or class B.

The signature only declares names, arities, and sorts. Concrete tensors and neural modules are assigned later through an `Interpretation`.

In [ ]:
import sltn
from sltn import Signature, Interpretation, KB

# Declare the signature
sig   = Signature("iris_clf")

sig.sort("Item")   # 4-feature flower sample

sig.predicate("A", ["Item"])  # class A
sig.predicate("B", ["Item"])  # class B
sig.predicate("C", ["Item"])  # class C
sig.predicate("T", [])

sig.variable("x_A", "Item")
sig.variable("x_B", "Item")
sig.variable("x_C", "Item")
sig.variable("x_AvB", "Item")
sig.variable("x_BvC", "Item")
sig.variable("x_AvC", "Item")
sig.variable("x", "Item")

print(sig)

## 2. Formulas and knowledge bases

We define several formulas and combine them into alternative knowledge bases. Each knowledge base corresponds to a different supervision regime.

### Full supervision

In the fully supervised case, each class has its own universally quantified formula:

$$
\forall x_A\, A(x_A), \qquad
\forall x_B\, B(x_B), \qquad
\forall x_C\, C(x_C).
$$

This states that all examples assigned to class A should satisfy predicate $A$, all examples assigned to class B should satisfy (B), and all examples assigned to class C should satisfy $C$.

### Supervision with a quantitative prior
The second knowledge base directly supervises classes A and B and uses a quantitative prior for class $C$:
$$T()\rightarrow \forall x : C(x).$$
Here $T$ is a nullary predicate grounded to a fixed truth value. This allows the knowledge base to include a soft estimate for class $C$ rather than direct class-specific supervision through $x_C$.

### Partial supervision with ambiguous labels

The third knowledge base uses disjunctions to express ambiguous supervision:
$$
\forall x_{A∨B}​(A(x_{A∨B}​)∨B(x_{A∨B​})),
$$
and analogously for $B \lor C$ and $A \lor C$. This means that an example may be known to belong to one of two possible classes without specifying which one exactly. The ambiguity is represented directly at the logical level through disjunction.

In [ ]:
phi_A = sig.parse("∀ x_A: A(x_A)")
phi_B = sig.parse("∀ x_B: B(x_B)")
phi_C = sig.parse("∀ x_C: C(x_C)")
phi_T = sig.parse("T() -> ∀ x: C(x)")
phi_AvB = sig.parse("∀ x_AvB: (A(x_AvB) ∨ B(x_AvB))")
phi_BvC = sig.parse("∀ x_BvC: (B(x_BvC) ∨ C(x_BvC))")
phi_AvC = sig.parse("∀ x_AvC: (A(x_AvC) ∨ C(x_AvC))")
phi_AvBvC = sig.parse("∀ x: (A(x) ∨ B(x) ∨ C(x))")

kb1 = KB([phi_A, phi_B, phi_C]) # full supervision
kb2 = KB([phi_A, phi_B, phi_T]) # full supervision of a subset of classesK with a quantitative estimation for the rest
kb3 = KB([phi_AvB, phi_BvC, phi_AvC]) # partial supervision with ambiguous labels
kbs = [kb1, kb2, kb3]

_kb = 2

phi_kb = kbs[_kb-1]
print(phi_kb)

The variable `_kb` selects which supervision regime is used for training:

- `_kb = 1`: full supervision;
- `_kb = 2`: supervision of classes A and B with a quantitative prior for class C;
- `_kb = 3`: partial supervision through ambiguous disjunctive labels.

The selected knowledge base is assigned to `phi_kb`.

## 3. Neural model

The three class predicates are grounded by a shared neural classifier. The model maps each four-dimensional Iris example to a probability vector over the three classes using a softmax output layer. The predicates `A`, `B`, and `C` are then interpreted as class-specific projections of this vector.
The softmax outputs are in $[0, 1]$, thus they can be used directly as fuzzy truth values for the corresponding predicates.


In [ ]:
class MLP(torch.nn.Module):
    """Returns logits over 3 classes from 4 input features."""
    def __init__(self, layer_sizes=(4, 16, 16, 8, 3)):
        super().__init__()
        self.elu = torch.nn.ELU()
        self.softmax = torch.nn.Softmax(dim=-1)
        self.dropout = torch.nn.Dropout(0.2)
        layers = []
        for i in range(1, len(layer_sizes)):
            layers.append(torch.nn.Linear(layer_sizes[i - 1], layer_sizes[i]))
        self.linear_layers = torch.nn.ModuleList(layers)

    def forward(self, x):
        for layer in self.linear_layers[:-1]:
            x = self.dropout(self.elu(layer(x))) if self.training else self.elu(layer(x))
        x = self.softmax(self.linear_layers[-1](x))
        return x

## 4. Interpretation

An `Interpretation` grounds the symbolic vocabulary declared in the signature.

In this notebook:

- the sort `Item` is interpreted as a tensor type with shape `(4,)`, corresponding to the four Iris features;
- predicates `A`, `B`, and `C` are grounded by selecting the corresponding output component of the shared softmax MLP;
- the nullary predicate `T` is grounded to a fixed truth value, used as a quantitative prior in the second knowledge base;
- variables are assigned tensors batch by batch during evaluation and training.

The interpretation also carries the fuzzy logic operators used to evaluate connectives, quantifiers, and knowledge-base satisfaction.

In [ ]:
interp = Interpretation(sig).to(device)

# ground sorts
R4 = sltn.Type("R4", shape=(4,))
interp["Item"] = R4
mlp = MLP([4, 8, 3]).to(device)


class ClassHead(torch.nn.Module):
    """Class-specific probability: column `k` of the shared MLP output."""
    def __init__(self, mlp, k):
        super().__init__()
        self.mlp, self.k = mlp, k

    def forward(self, x):
        return self.mlp(x)[..., [self.k]]


# A trainable grounding must be an nn.Module, so the shared MLP is registered once
interp["A"] = ClassHead(mlp, 0)
interp["B"] = ClassHead(mlp, 1)
interp["C"] = ClassHead(mlp, 2)
interp["T"] = lambda : torch.tensor([0.33]).to(device)  # quantitative estimation for the unobserved class
print(interp)

## 5. Data loaders and metrics

In [ ]:
class DataLoader:
    def __init__(self, data, labels, batch_size=64, shuffle=True):
        self.data = data.to(device)
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        idxlist = list(range(self.data.shape[0]))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, self.data.shape[0], self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield self.data[idx], self.labels[idx]


def compute_sat_level(loader, interp):
    mean_sat = 0
    for data, labels in loader:
        interp["x_A"] = data[labels == 0]
        interp["x_B"] = data[labels == 1]
        interp["x_C"] = data[labels == 2]
        interp["x_AvB"] = data[labels != 2]
        interp["x_AvC"] = data[labels != 1]
        interp["x_BvC"] = data[labels != 0]
        interp["x"] = data
        mean_sat += interp(phi_kb).value
    return mean_sat / len(loader)


def compute_accuracy(loader, interp,print_preds=False):
    mean_acc = 0.0
    for data, labels in loader:
        with torch.no_grad():
            interp["x"] = data
            preds = torch.concat([interp(sig.parse("A(x)")).value,
                                 interp(sig.parse("B(x)")).value,
                                 interp(sig.parse("C(x)")).value], dim=1).cpu().numpy()
        mean_acc += accuracy_score(labels.cpu().numpy(), np.argmax(preds, axis=1))
        if print_preds:
            for pred, label in zip(preds, labels):
                print(torch.argmax(pred),label,torch.argmax(pred) == label)
    return mean_acc / len(loader)

train_loader = DataLoader(train_data, train_labels, 12)
test_loader  = DataLoader(test_data,  test_labels, shuffle=False)

## 6. Training loop

Training maximizes the satisfaction of the selected knowledge base.

For each batch:

1. the batch is split according to the labels required by the formulas;
2. the corresponding logical variables are grounded in the interpretation;
3. the selected knowledge base is evaluated under the current neural grounding;
4. the loss is defined as: $\mathcal{L} = 1 - \mathrm{SatAgg}(\mathcal{K})$;
5. gradients are backpropagated through the fuzzy evaluation of the formulas and through the neural predicate groundings;
6. the optimizer updates the parameters of the shared MLP.
The printed metrics report the average loss, the satisfaction level of the selected knowledge base, and the classification accuracy on the training and test splits.

In [ ]:
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.001)
nr_of_epochs = 3000 if _kb == 1 else 20000 if _kb == 2 else 3000
for epoch in range(nr_of_epochs):
    train_loss = 0.0
    mlp.train()
    for data, labels in train_loader:
        for P,i in {"A": 0, "B": 1, "C": 2}.items():
            interp[f"x_{P}"] = data[labels == i]
        interp["x_AvB"] = data[labels <= 1]
        interp["x_BvC"] = data[labels >= 1]
        interp["x_AvC"] = data[labels != 1]
        interp['x'] = data

        optimizer.zero_grad()
        loss = 1 - interp(phi_kb).value
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
    train_loss /= len(train_loader)

    if epoch % 50 == 0 or epoch == nr_of_epochs - 1:
        print(f"epoch {epoch:3d} | loss {train_loss:.4f} "
              f"| Train Sat {compute_sat_level(train_loader, interp).item():.3f} "
              f"| Test Sat {compute_sat_level(test_loader, interp).item():.3f} "
              f"| Train Acc {compute_accuracy(train_loader, interp):.3f} "
              f"| Test Acc {compute_accuracy(test_loader, interp):.3f}")